# Assignment 1 — Multimodal Image Classification


In [1]:
# Setup: imports and helper installs (run once)
# If you need to install missing libraries, uncomment the pip lines below.
# Note: this environment may not have internet access; run pip installs locally if needed.

# !pip install pycocotools opencv-python gensim scikit-learn matplotlib seaborn tqdm pillow

import os
import json
from pathlib import Path
from tqdm import tqdm
import numpy as np
import cv2
from PIL import Image
from gensim.models import Word2Vec, KeyedVectors
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import matplotlib.pyplot as plt
import seaborn as sns
import pickle

In [2]:
COCO_ROOT = '/home/BTECH_7TH_SEM/MS-COCO'  
IM_DIR = os.path.join(COCO_ROOT, 'val2017')
CAPTIONS_JSON = os.path.join(COCO_ROOT,'annotations_trainval2017', 'annotations', 'captions_val2017.json')
INSTANCES_JSON = os.path.join(COCO_ROOT, 'annotations_trainval2017', 'annotations', 'instances_val2017.json')

# --- Utility loader ---
def load_coco_captions(captions_json):
    with open(captions_json, 'r', encoding='utf-8') as f:
        data = json.load(f)
    # Build image_id -> list of captions mapping
    img2caps = {}
    for item in data['annotations']:
        img_id = item['image_id']
        img2caps.setdefault(img_id, []).append(item['caption'])
    return img2caps

def load_image_labels(instances_json):
    # Build image_id -> list of category_ids
    with open(instances_json, 'r', encoding='utf-8') as f:
        data = json.load(f)
    img2cat = {}
    catid2name = {c['id']:c['name'] for c in data['categories']}
    for ann in data['annotations']:
        img_id = ann['image_id']
        cat_id = ann['category_id']
        img2cat.setdefault(img_id, []).append(cat_id)
    # Convert to most frequent category per image (simple heuristic)
    img2label = {}
    for img_id, cats in img2cat.items():
        most = max(set(cats), key=cats.count)
        img2label[img_id] = most
    return img2label, catid2name

In [3]:
# ---------- Feature extraction helpers ----------
import re
def preprocess_caption(c):
    # lowercase, remove non-alpha, simple tokenization
    c = c.lower()
    c = re.sub(r"[^a-z0-9\s]", '', c)
    tokens = c.split()
    return tokens

def extract_canny_features(image_path, resize=(224,224), sigma=1.0):
    # Load image as grayscale, resize, apply Canny, return flattened histogram of edges
    img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        raise FileNotFoundError(image_path)
    img = cv2.resize(img, resize)
    # Blur for stability
    blur = cv2.GaussianBlur(img, (5,5), sigma)
    edges = cv2.Canny(blur, 100, 200)
    # compute simple feature: normalized histogram of edge pixels by row/col or flatten
    # Here we'll use a small downsampled edge map as feature (e.g., 28x28)
    small = cv2.resize(edges, (28,28)).astype(np.float32)/255.0
    feat = small.flatten()  # 784-dim
    return feat

def caption_to_embedding(caption_tokens, w2v_model, dim=300, agg='avg'):
    # caption_tokens: list of tokens
    vecs = []
    for t in caption_tokens:
        if t in w2v_model.wv:
            vecs.append(w2v_model.wv[t])
    if len(vecs)==0:
        return np.zeros(dim, dtype=np.float32)
    vecs = np.stack(vecs)
    if agg=='avg':
        return vecs.mean(axis=0)
    elif agg=='max':
        return vecs.max(axis=0)
    else:
        return vecs.mean(axis=0)


In [4]:
# Build dataset features (this may take time). We'll demonstrate with a small subset.
def build_features(img2caps, img2label, image_dir, max_images=None, w2v_model=None):
    X_img = []
    X_txt = []
    y = []
    img_ids = list(img2caps.keys())
    if max_images:
        img_ids = img_ids[:max_images]
    missing = 0
    for img_id in tqdm(img_ids):
        img_fname = f'{img_id:012d}.jpg'
        img_path = os.path.join(image_dir, img_fname)
        try:
            img_feat = extract_canny_features(img_path)
        except FileNotFoundError:
            missing += 1
            continue
        # captions -> aggregate embedding: average over 5 captions
        caps = img2caps.get(img_id, [])
        cap_embs = []
        for c in caps:
            toks = preprocess_caption(c)
            if w2v_model:
                cap_embs.append(caption_to_embedding(toks, w2v_model))
        if len(cap_embs)==0:
            # fallback zeros
            cap_emb = np.zeros(w2v_model.vector_size if w2v_model else 300)
        else:
            cap_emb = np.stack(cap_embs).mean(axis=0)
        label = img2label.get(img_id, None)
        if label is None:
            continue
        X_img.append(img_feat)
        X_txt.append(cap_emb)
        y.append(label)
    print('Missing images:', missing)
    X_img = np.array(X_img)
    X_txt = np.array(X_txt)
    y = np.array(y)
    return X_img, X_txt, y


In [5]:
def train_w2v_on_captions(img2caps, vector_size=300, window=5, min_count=2, epochs=10):
    sentences = []
    for caps in img2caps.values():
        for c in caps:
            sentences.append(preprocess_caption(c))
    print('Training Word2Vec on', len(sentences), 'sentences...')
    model = Word2Vec(sentences, vector_size=vector_size, window=window, min_count=min_count, epochs=epochs)
    return model

In [6]:
# Fusion: concatenation
def fuse_concat(X_img, X_txt):
    return np.concatenate([X_img, X_txt], axis=1)

# Small pipeline to train and evaluate logistic regression
def train_eval(X, y, test_size=0.2, val_size=0.1, random_state=42):
    X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=test_size, stratify=y, random_state=random_state)
    relative_val = val_size / (1 - test_size)
    X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=relative_val, stratify=y_temp, random_state=random_state)
    print('Train/Val/Test sizes:', X_train.shape[0], X_val.shape[0], X_test.shape[0])
    clf = LogisticRegression(max_iter=1000, multi_class='multinomial', solver='saga', n_jobs=-1)
    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    cm = confusion_matrix(y_test, y_pred)
    return clf, acc, cm, (X_train, X_val, X_test, y_train, y_val, y_test)


In [7]:
# Conduct unimodal vs multimodal experiments
def run_experiments(X_img, X_txt, y):
    results = {}
    # Image-only
    clf_i, acc_i, cm_i, splits_i = train_eval(X_img, y)
    results['image_only'] = {'clf':clf_i, 'acc':acc_i, 'cm':cm_i}
    # Text-only
    clf_t, acc_t, cm_t, splits_t = train_eval(X_txt, y)
    results['text_only'] = {'clf':clf_t, 'acc':acc_t, 'cm':cm_t}
    # Multimodal (concat)
    X_fused = fuse_concat(X_img, X_txt)
    clf_m, acc_m, cm_m, splits_m = train_eval(X_fused, y)
    results['multimodal'] = {'clf':clf_m, 'acc':acc_m, 'cm':cm_m}
    return results

# Visualization helper for confusion matrix
def plot_confusion(cm, title='Confusion Matrix', labels=None):
    plt.figure(figsize=(8,6))
    sns.heatmap(cm, annot=False, fmt='d', cmap='Blues')
    plt.title(title)
    if labels:
        plt.xticks(np.arange(len(labels))+0.5, labels, rotation=90)
        plt.yticks(np.arange(len(labels))+0.5, labels, rotation=0)
    plt.show()


In [8]:
# Save models/results
def save_pickle(obj, path):
    with open(path, 'wb') as f:
        pickle.dump(obj, f)


In [9]:
# Add this cell after all the function definitions to actually run the pipeline

# Step 1: Load the data
print("Step 1: Loading COCO data...")
img2caps = load_coco_captions(CAPTIONS_JSON)
img2label, catid2name = load_image_labels(INSTANCES_JSON)
print(f'Loaded captions for {len(img2caps)} images')
print(f'Loaded labels for {len(img2label)} images')
print('Example categories:', list(catid2name.values())[:10])

# Step 2: Train Word2Vec model on captions
print("\nStep 2: Training Word2Vec model...")
w2v_model = train_w2v_on_captions(img2caps, vector_size=300, epochs=10)
print(f'Word2Vec vocabulary size: {len(w2v_model.wv)}')

# Step 3: Build features (start with small subset for testing)
print("\nStep 3: Extracting features...")
# Start with 2000 images for faster testing - increase this number for full experiment
MAX_IMAGES = None  # Change to None for all images
X_img, X_txt, y = build_features(img2caps, img2label, IM_DIR, 
                                max_images=MAX_IMAGES, w2v_model=w2v_model)

print(f'Image features shape: {X_img.shape}')
print(f'Text features shape: {X_txt.shape}')
print(f'Labels shape: {y.shape}')
print(f'Number of unique classes: {len(np.unique(y))}')

# Check class distribution before running experiments
unique_labels, counts = np.unique(y, return_counts=True)
print(f'Class distribution:')
print(f'  Min samples per class: {counts.min()}')
print(f'  Max samples per class: {counts.max()}')
print(f'  Mean samples per class: {counts.mean():.1f}')
print(f'  Classes with <5 samples: {sum(counts < 5)}')

# Filter out classes with very few samples to avoid stratification issues
min_samples = 5  # Require at least 5 samples per class
valid_classes = unique_labels[counts >= min_samples]
if len(valid_classes) < len(unique_labels):
    print(f'\nFiltering dataset: keeping {len(valid_classes)}/{len(unique_labels)} classes')
    mask = np.isin(y, valid_classes)
    X_img = X_img[mask]
    X_txt = X_txt[mask]
    y = y[mask]
    print(f'Dataset size after filtering: {y.shape[0]} samples, {len(np.unique(y))} classes')

# Step 4: Run all experiments (unimodal vs multimodal)
print("\nStep 4: Running experiments...")
results = run_experiments(X_img, X_txt, y)

# Step 5: Display results
print("\n" + "="*50)
print("RESULTS SUMMARY")
print("="*50)
print(f"Image-only accuracy: {results['image_only']['acc']:.4f}")
print(f"Text-only accuracy:  {results['text_only']['acc']:.4f}")
print(f"Multimodal accuracy: {results['multimodal']['acc']:.4f}")

# Calculate improvement
img_acc = results['image_only']['acc']
txt_acc = results['text_only']['acc']
mm_acc = results['multimodal']['acc']
best_uni = max(img_acc, txt_acc)
improvement = ((mm_acc - best_uni) / best_uni) * 100

print(f"\nMultimodal improvement over best unimodal: {improvement:+.2f}%")

# Step 6: Visualize confusion matrices (optional - may be too large for many classes)
print("\nStep 6: Generating confusion matrix visualizations...")

# Only plot if we have reasonable number of classes (< 20)
n_classes = len(np.unique(y))
if n_classes <= 20:
    # Get class names for the labels we actually have
    unique_labels = np.unique(y)
    class_names = [catid2name.get(label, f'Class_{label}') for label in unique_labels]
    
    plot_confusion(results['image_only']['cm'], 'Image-only Confusion Matrix', class_names)
    plot_confusion(results['text_only']['cm'], 'Text-only Confusion Matrix', class_names)
    plot_confusion(results['multimodal']['cm'], 'Multimodal Confusion Matrix', class_names)
else:
    print(f"Too many classes ({n_classes}) for confusion matrix visualization")
    print("Confusion matrices computed but not plotted")

# Step 7: Save models and results
print("\nStep 7: Saving models...")
save_pickle(results['multimodal']['clf'], 'assignment1/multimodal_classifier.pkl')
save_pickle(w2v_model, 'assignment1/word2vec_model.pkl')
save_pickle(results, 'assignment1/experiment_results.pkl')

print("\nPipeline completed successfully!")
print("\nSaved files:")
print("- multimodal_classifier.pkl")
print("- word2vec_model.pkl") 
print("- experiment_results.pkl")

# Optional: Detailed analysis
print("\n" + "="*50)
print("DETAILED ANALYSIS")
print("="*50)

# Feature statistics
print(f"Image feature statistics:")
print(f"  Mean: {X_img.mean():.4f}, Std: {X_img.std():.4f}")
print(f"  Min: {X_img.min():.4f}, Max: {X_img.max():.4f}")

print(f"Text feature statistics:")
print(f"  Mean: {X_txt.mean():.4f}, Std: {X_txt.std():.4f}")
print(f"  Min: {X_txt.min():.4f}, Max: {X_txt.max():.4f}")

# Class distribution
unique_labels, counts = np.unique(y, return_counts=True)
print(f"\nClass distribution:")
for label, count in zip(unique_labels[:10], counts[:10]):  # Show top 10
    class_name = catid2name.get(label, f'Class_{label}')
    print(f"  {class_name}: {count} samples")
if len(unique_labels) > 10:
    print(f"  ... and {len(unique_labels) - 10} more classes")

Step 1: Loading COCO data...
Loaded captions for 5000 images
Loaded labels for 4952 images
Example categories: ['person', 'bicycle', 'car', 'motorcycle', 'airplane', 'bus', 'train', 'truck', 'boat', 'traffic light']

Step 2: Training Word2Vec model...
Training Word2Vec on 25014 sentences...
Word2Vec vocabulary size: 4288

Step 3: Extracting features...


100%|██████████| 5000/5000 [01:13<00:00, 67.77it/s]
/home/BTECH_7TH_SEM/Desktop/MML-RL-and-NLP/MML/mml-venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


Missing images: 0
Image features shape: (4952, 784)
Text features shape: (4952, 300)
Labels shape: (4952,)
Number of unique classes: 80
Class distribution:
  Min samples per class: 1
  Max samples per class: 1816
  Mean samples per class: 61.9
  Classes with <5 samples: 8

Filtering dataset: keeping 72/80 classes
Dataset size after filtering: 4928 samples, 72 classes

Step 4: Running experiments...
Train/Val/Test sizes: 3449 493 986
Train/Val/Test sizes: 3449 493 986


/home/BTECH_7TH_SEM/Desktop/MML-RL-and-NLP/MML/mml-venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


Train/Val/Test sizes: 3449 493 986


/home/BTECH_7TH_SEM/Desktop/MML-RL-and-NLP/MML/mml-venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(



RESULTS SUMMARY
Image-only accuracy: 0.2961
Text-only accuracy:  0.5456
Multimodal accuracy: 0.4523

Multimodal improvement over best unimodal: -17.10%

Step 6: Generating confusion matrix visualizations...
Too many classes (72) for confusion matrix visualization
Confusion matrices computed but not plotted

Step 7: Saving models...

Pipeline completed successfully!

Saved files:
- multimodal_classifier.pkl
- word2vec_model.pkl
- experiment_results.pkl

DETAILED ANALYSIS
Image feature statistics:
  Mean: 0.0717, Std: 0.1803
  Min: 0.0000, Max: 1.0000
Text feature statistics:
  Mean: 0.0057, Std: 0.1753
  Min: -1.0518, Max: 1.0214

Class distribution:
  person: 1816 samples
  bicycle: 22 samples
  car: 218 samples
  motorcycle: 25 samples
  airplane: 53 samples
  bus: 20 samples
  train: 55 samples
  truck: 62 samples
  boat: 53 samples
  traffic light: 80 samples
  ... and 62 more classes
